In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam

def load_data(path, width, height, batch_size=16):
    datagen = ImageDataGenerator(
        horizontal_flip=True,
        vertical_flip=True,
        rescale=1/255.,
        validation_split=0.2)

    train_flow = datagen.flow_from_directory(
        path,
        target_size=(width, height),
        batch_size=batch_size,
        class_mode='sparse',
        subset='training',
        seed=12345)

    val_flow = datagen.flow_from_directory(
        path,
        target_size=(width, height),
        batch_size=batch_size,
        class_mode='sparse',
        subset='validation',
        seed=12345)

    return train_flow, val_flow

I0000 00:00:1778821687.877258   46413 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1778821687.902460   46413 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1778821688.463307   46413 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
def create_resnet_model(input_shape):
    backbone = ResNet50(input_shape=input_shape, weights='imagenet', include_top=False)
    
    # usaremos un modelo previamente entrenado con sus parámetros susceptibles de ser aprendidos intactos
    for layer in backbone.layers:
        layer.trainable = False
    
    # entrenemos la parte de clasificación
    model = Sequential()
    model.add(backbone)
    model.add(GlobalAveragePooling2D())
    model.add(Dense(128, activation='relu'))
    model.add(Dense(15, activation='softmax'))
    
    optimizer = Adam(learning_rate=0.001)
    model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

In [3]:
def train_model(model, train_data, val_data, epochs=10):

    model.fit(
        train_data,
        validation_data=val_data,
        epochs=epochs,
        verbose=1
    )
    return model

In [ ]:
## Parámetros
# Dependiendo de tu máquina, puedes jugar con los parámetros de ancho y alto
# para un entrenamiento más rápido
width = 128
height = 128
batch_size = 32
input_shape = (width, height,3)

path = '../data/archive'
# Carga los datos de entrenamiento y validación
train_gen, val_gen = load_data(path, width, height , batch_size)

# Crea y entrena el modelo simple
model = create_resnet_model(input_shape)
trained_model = train_model(model, train_gen, val_gen, epochs=10)

Found 56445 images belonging to 15 classes.
Found 14104 images belonging to 15 classes.


W0000 00:00:1778821699.743759   46413 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Epoch 1/10


I0000 00:00:1778821700.343512   46413 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.


  17/3528 ━━━━━━━━━━━━━━━━━━━━ 2:37 45ms/step - accuracy: 0.1902 - loss: 2.5464